In [ ]:
import multiprocessing as mp
from dataclasses import dataclass
from datetime import datetime
from enum import IntEnum
from pathlib import Path
from typing import Callable, final, override

import gdown
import lightning as L
import pandas as pd
import timm
import timm.data
import torch
import torch.nn as nn
from lightning.pytorch.loggers import WandbLogger
from PIL import Image
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader, Dataset

SEED = 67
L.seed_everything(SEED, workers=True)


class Stat(IntEnum):
    good = 0
    bad = 1


@final
@dataclass(frozen=True, slots=True)
class Config:
    epochs_phase1: int = 5  # Phase 1: 凍結主幹，僅訓練 Head
    epochs_phase2: int = 15  # Phase 2: 解凍主幹最後幾層微調
    lr: float = 1e-3  # Phase 1 使用較大 LR
    weight_decay: float = 1e-2


@final
@dataclass(frozen=True, slots=True)
class DataConfig:
    data_dir: Path = Path("data")
    train_dir: Path = data_dir / "train_dataset"
    test_dir: Path = data_dir / "test_dataset"
    pin_memory: bool = True
    image_size: int = 1024
    batch_size: int = 32
    test_batch_size: int = 128  # maximum batch size that can fit in GPU memory
    num_workers: int = mp.cpu_count()
    n_folds: int = 5


tcfg = Config()
dcfg = DataConfig()

In [ ]:
@final
class TrainingDataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform: Callable):
        super().__init__()
        self.df = df
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    @override
    def __getitem__(self, index: int) -> tuple[torch.Tensor, str, float]:
        row = self.df.iloc[index]
        img = Image.open(row["path"]).convert("RGB")
        img = self.transform(img)
        return img, row["comp"], float(row["stat"])

In [ ]:
@final
class InspladDataModule(L.LightningDataModule):
    def __init__(
        self, train_transforms: Callable, test_transforms: Callable, fold_idx: int
    ):
        super().__init__()
        self.train_transforms = train_transforms
        self.test_transforms = test_transforms
        self.fold_idx = fold_idx

        self.train_ds: TrainingDataset
        self.val_ds: TrainingDataset

    @override
    def prepare_data(self):
        train_file_id = r"14B3Jsj4DzoCrMC4YXlXEp0ej_reMgWkq"
        train_checksum = r"md5:449e4617aefa0d9c9d059e21c38b32f5"
        test_file_id = r"1RhPBNwWxRYK0M8UvQcBBnVSpPzsEF3xn"
        test_checksum = r"md5:5edc01fa26e9563449aa7e7885242e71"

        gdown.cached_download(
            id=train_file_id,
            path=f"{dcfg.train_dir}.zip",
            hash=train_checksum,
        )
        gdown.cached_download(
            id=test_file_id,
            path=f"{dcfg.test_dir}.zip",
            hash=test_checksum,
        )
        gdown.extractall(f"{dcfg.train_dir}.zip")
        gdown.extractall(f"{dcfg.test_dir}.zip")

    @override
    def setup(self, stage: str | None = None) -> None:
        if stage == "fit":
            # Unlovely implementation, but I believe this is the only way to do cross validations with DataModule
            full_df = pd.DataFrame({"path": sorted(dcfg.train_dir.rglob("*.jpg"))})
            full_df["comp"] = full_df["path"].apply(lambda p: p.parents[1].name)
            full_df["stat"] = full_df["path"].apply(lambda p: Stat[p.parent.name])

            skf = StratifiedKFold(
                n_splits=dcfg.n_folds, shuffle=True, random_state=SEED
            )
            stratify_col = full_df["comp"] + "_" + full_df["stat"].astype(str)
            train_idx, val_idx = list(skf.split(full_df, stratify_col))[self.fold_idx]

            train_df = full_df.iloc[train_idx]
            val_df = full_df.iloc[val_idx]
            self.train_ds = TrainingDataset(train_df, self.train_transforms)
            self.val_ds = TrainingDataset(val_df, self.test_transforms)

    @override
    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=dcfg.batch_size,
            shuffle=True,
            num_workers=dcfg.num_workers,
            pin_memory=dcfg.pin_memory,
        )

    @override
    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=dcfg.test_batch_size,
            shuffle=False,
            num_workers=dcfg.num_workers,
            pin_memory=dcfg.pin_memory,
        )

    # def test_dataloader(self):
    #     return DataLoader(
    #         self.test_ds,
    #         batch_size=dcfg.test_batch_size,
    #         shuffle=False,
    #         num_workers=dcfg.num_workers,
    #         pin_memory=dcfg.pin_memory,
    #     )

In [ ]:
@final
class MultiHeadDino(L.LightningModule):
    def __init__(self, dino: str = "vit_large_patch16_dinov3_qkvb.lvd1689m"):
        super().__init__()
        self.save_hyperparameters(
            {
                "dino": dino,
                "epochs_phase1": tcfg.epochs_phase1,
                "epochs_phase2": tcfg.epochs_phase2,
                "lr": tcfg.lr,
                "weight_decay": tcfg.weight_decay,
                "image_size": dcfg.image_size,
                "batch_size": dcfg.batch_size,
                "test_batch_size": dcfg.test_batch_size,
                "n_folds": dcfg.n_folds,
                "seed": SEED,
            }
        )
        # 建議使用 pos_weight，你可以先設個通用值，或針對 5 個器材各給一個權重
        self.criterion = nn.BCEWithLogitsLoss()
        self.backbone = timm.create_model(
            dino, pretrained=True, num_classes=0, dynamic_img_size=True
        )

        for param in self.backbone.parameters():
            param.requires_grad = False

        embed_dim = self.backbone.num_features
        self.heads = nn.ModuleDict(
            {
                "glass-insulator": nn.Linear(embed_dim, 1),
                "lightning-rod-suspension": nn.Linear(embed_dim, 1),
                "polymer-insulator-upper-shackle": nn.Linear(embed_dim, 1),
                "vari-grip": nn.Linear(embed_dim, 1),
                "yoke-suspension": nn.Linear(embed_dim, 1),
            }
        )

        # 用於收集 Validation 每個 batch 的輸出
        self.val_step_outputs = []

    @override
    def forward(self, img: torch.Tensor, comp: list[str]):
        features = self.backbone(img)
        outputs = []
        for f, c in zip(features, comp):
            outputs.append(self.heads[c](f))
        return torch.stack(outputs)

    def training_step(self, batch, batch_idx):
        inputs, comps, target = batch
        target = target.unsqueeze(1)  # 修正：[B] ->[B, 1] 匹配輸出形狀
        output = self(inputs, comps)
        loss = self.criterion(output, target)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        inputs, comps, target = batch
        output = self(inputs, comps)
        # 儲存 Logit 機率與真實標籤，放到 Epoch End 計算
        self.val_step_outputs.append(
            {
                "comps": comps,
                "preds": torch.sigmoid(output).detach().cpu(),
                "targets": target.cpu(),
            }
        )

    def on_validation_epoch_end(self):
        # 攤平所有 batch 的數據
        all_comps = [c for batch in self.val_step_outputs for c in batch["comps"]]
        all_preds = (
            torch.cat([batch["preds"] for batch in self.val_step_outputs])
            .squeeze()
            .numpy()
        )
        all_targets = torch.cat(
            [batch["targets"] for batch in self.val_step_outputs]
        ).numpy()

        df = pd.DataFrame({"comp": all_comps, "pred": all_preds, "target": all_targets})

        # 針對 5 個器材獨立尋找最佳閾值
        for comp_name in df["comp"].unique():
            comp_df = df[df["comp"] == comp_name]
            if (
                len(comp_df["target"].unique()) > 1
            ):  # 確保該器材在 Validation 中同時有正負樣本
                precisions, recalls, thresholds = precision_recall_curve(
                    comp_df["target"], comp_df["pred"]
                )

                # 尋找 Precision >= 92% (留2%容錯給測試集) 時的最大 Recall
                valid_idx = precisions >= 0.92
                if valid_idx.any():
                    valid_indices = valid_idx.nonzero()[0]
                    best_idx = valid_indices[recalls[valid_indices].argmax()]
                    best_recall = float(recalls[best_idx])
                    best_precision = float(precisions[best_idx])
                    # 紀錄到 TensorBoard/Logger 中
                    self.log(
                        f"val_precision_{comp_name}", best_precision, sync_dist=True
                    )
                    self.log(f"val_recall_{comp_name}", best_recall, sync_dist=True)

        self.val_step_outputs.clear()  # 清空記憶體

    def configure_optimizers(self):
        # 最佳實踐：使用 AdamW 搭配 Cosine Annealing
        optimizer = torch.optim.AdamW(
            self.parameters(), lr=tcfg.lr, weight_decay=tcfg.weight_decay
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=tcfg.epochs_phase1
        )
        return [optimizer], [scheduler]

    def build_transforms(self):
        data_config = timm.data.resolve_model_data_config(self.backbone)
        # 強迫修改尺寸為 H200 等級的高解析度
        data_config["input_size"] = (3, dcfg.image_size, dcfg.image_size)
        train_transform = timm.data.create_transform(**data_config, is_training=True)
        test_transform = timm.data.create_transform(**data_config, is_training=False)
        return train_transform, test_transform

In [ ]:
group_name = datetime.now().strftime("%m%d_%H%M")

models = []
for fold_idx in range(dcfg.n_folds):
    model = MultiHeadDino()
    dm = InspladDataModule(*model.build_transforms(), fold_idx)

    wandb_logger = WandbLogger(
        project="InsPLAD",
        entity="wytsai7660",
        name=f"fold-{fold_idx}",
        group=group_name,
    )
    trainer = L.Trainer(logger=wandb_logger)
    trainer.fit(model, datamodule=dm)
    models.append(model)

In [ ]:
# testing